In [1]:
import pandas as pd
import numpy as np

# Load datasets
ev_india = pd.read_csv("EVIndia.csv")
ev_specs = pd.read_csv("ElectricCarData_Clean.csv")
charging = pd.read_csv("electric_vehicle_charging_station_list.csv")
cheap_ev = pd.read_csv("Cheapestelectriccars-EVDatabase (1).csv")

In [2]:
print(ev_india.head())
print(ev_india.info())
print(ev_india.isnull().sum())

                     Car             Style               Range Transmission  \
0          Tata Nexon EV       Compact SUV  312 Km/Full Charge    Automatic   
1          Tata Tigor EV  Subcompact Sedan  306 Km/Full Charge    Automatic   
2      Tata Nexon EV Max       Compact SUV  437 Km/Full Charge    Automatic   
3               MG ZS EV       Compact SUV  419 Km/Full Charge    Automatic   
4  Hyundai Kona Electric       Compact SUV  452 Km/Full Charge    Automatic   

  VehicleType  PriceRange(Lakhs)   Capacity BootSpace           BaseModel  \
0    Electric              17.40  5 Seater      350 L                 XM    
1    Electric              13.64  5 Seater      316 L                 XE    
2    Electric              19.24  5 Seater      350 L     XZ Plus 3.3 kW    
3    Electric              25.88  5 Seater      448 L             Excite    
4    Electric              23.98  5 Seater         na  Premium Dual Tone    

              TopModel  
0    Dark XZ Plus LUX   
1   XZ Plus 

In [3]:
# Fill numeric missing values with median
ev_india.fillna(ev_india.median(numeric_only=True), inplace=True)

# Fill categorical with mode
for col in ev_india.select_dtypes(include='object'):
    ev_india[col].fillna(ev_india[col].mode()[0], inplace=True)

C:\Users\raksha\AppData\Local\Temp\ipykernel_288\154151640.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ev_india[col].fillna(ev_india[col].mode()[0], inplace=True)


In [4]:
ev_india.drop_duplicates(inplace=True)

In [5]:
ev_india.columns = ev_india.columns.str.strip().str.lower().str.replace(" ", "_")

In [6]:
# Step 1: Extract only numbers
ev_india['range_km'] = ev_india['range'].str.extract('(\d+)')

# Step 2: Convert to numeric
ev_india['range_km'] = pd.to_numeric(ev_india['range_km'])

In [7]:
print(ev_india.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   car                12 non-null     object 
 1   style              12 non-null     object 
 2   range              12 non-null     object 
 3   transmission       12 non-null     object 
 4   vehicletype        12 non-null     object 
 5   pricerange(lakhs)  12 non-null     float64
 6   capacity           12 non-null     object 
 7   bootspace          12 non-null     object 
 8   basemodel          12 non-null     object 
 9   topmodel           12 non-null     object 
 10  range_km           11 non-null     float64
dtypes: float64(2), object(9)
memory usage: 1.2+ KB
None


In [8]:
# Cost per km (used in Scenario 3)
ev_india['cost_per_km'] = ev_india['pricerange(lakhs)'] / ev_india['range_km']

# Efficiency category
ev_india['efficiency'] = np.where(ev_india['range_km'] > 400, 'High',
                         np.where(ev_india['range_km'] > 250, 'Medium', 'Low'))

In [9]:
charging.columns = charging.columns.str.lower().str.replace(" ", "_")

# Convert time if exists
# charging['time'] = pd.to_datetime(charging['time'], errors='coerce')

# Extract hour for peak analysis
# charging['hour'] = charging['time'].dt.hour

In [10]:
ev_india.to_csv("clean_ev_india.csv", index=False)
charging.to_csv("clean_charging.csv", index=False)